In [2]:
# CO-OCCURRENCE ANALYSIS (Development)
print("DAY 10: GENE CO-OCCURRENCE")
print("%" * 50)

import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# Load clean mutations (correct denominator)
mutations = pd.read_csv('../data/clean_mutations.tsv', sep='\t')

# Top 15 genes (from Day 8 frequencies)
non_hyper_mut = mutations[mutations['hypermutator'] != 1]
total_samples = non_hyper_mut['SAMPLE_ID'].nunique()
print(f" Denominator: {total_samples} samples with mutations")

# Binary matrix: 1=mutated (any non-silent), 0=WT
top_genes = non_hyper_mut.groupby('Hugo_Symbol')['SAMPLE_ID'].nunique().sort_values(ascending=False).head(15).index
print(f"Testing top {len(top_genes)} genes: {list(top_genes)}")

binary = non_hyper_mut[non_hyper_mut['Hugo_Symbol'].isin(top_genes)]
binary_pivot = binary.pivot_table(
    index='SAMPLE_ID', columns='Hugo_Symbol', 
    values='Variant_Classification', aggfunc='size', fill_value=0
).gt(0).astype(int)  # 1=mutated, 0=WT


all_samples = non_hyper_mut['SAMPLE_ID'].unique()
binary_pivot = binary_pivot.reindex(all_samples, fill_value=0)

print(f"Binary matrix: {binary_pivot.shape} ({len(binary_pivot)} samples × {len(top_genes)} genes)")
print("\nTP53 mutation rate:", binary_pivot['TP53'].mean()*100, "%")


DAY 10: GENE CO-OCCURRENCE
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
 Denominator: 504 samples with mutations
Testing top 15 genes: ['TP53', 'TTN', 'FAT1', 'CDKN2A', 'FRG1B', 'CSMD3', 'MUC16', 'PIK3CA', 'NOTCH1', 'SYNE1', 'LRP1B', 'KMT2D', 'PCLO', 'FLG', 'DNAH5']
Binary matrix: (504, 15) (504 samples × 15 genes)

TP53 mutation rate: 70.63492063492063 %


In [4]:
print("\nTEST 1: TP53 vs CDKN2A (predicted mutual exclusivity)")

tp53 = binary_pivot['TP53']
cdkn2a = binary_pivot['CDKN2A']

# 2x2 contingency table
a = ((tp53 == 1) & (cdkn2a == 1)).sum()      # both
b = ((tp53 == 1) & (cdkn2a == 0)).sum()      # TP53 only  
c = ((tp53 == 0) & (cdkn2a == 1)).sum()      # CDKN2A only
d = ((tp53 == 0) & (cdkn2a == 0)).sum()      # neither

table = [[a, b], [c, d]]
odds_ratio, p_value = fisher_exact(table)

print(f"Contingency table:")
print(f"  TP53+ CDKN2A+: {a:3d}  TP53+ CDKN2A-: {b:4d}")
print(f"  TP53- CDKN2A+: {c:3d}  TP53- CDKN2A-: {d:4d}")
print(f"Total: {a+b+c+d} = {total_samples} ✓")
print(f"\nOdds ratio: {odds_ratio:.3f}")
print(f"P-value:    {p_value:.2e}")
print(f"Direction:  {'Mutual exclusivity' if odds_ratio < 1 else 'Co-occurrence'}")


TEST 1: TP53 vs CDKN2A (predicted mutual exclusivity)
Contingency table:
  TP53+ CDKN2A+: 105  TP53+ CDKN2A-:  251
  TP53- CDKN2A+:   5  TP53- CDKN2A-:  143
Total: 504 = 504 ✓

Odds ratio: 11.964
P-value:    1.05e-12
Direction:  Co-occurrence


In [5]:
# TEST 2: ALL PAIRWISE TESTS
print("\nTEST 2: ALL {len(top_genes)}C2 = {len(top_genes)*(len(top_genes)-1)//2} pairs")

results = []
for g1, g2 in combinations(top_genes, 2):
    col1 = binary_pivot[g1]
    col2 = binary_pivot[g2]
    
    a = ((col1 == 1) & (col2 == 1)).sum()
    b = ((col1 == 1) & (col2 == 0)).sum()
    c = ((col1 == 0) & (col2 == 1)).sum()
    d = ((col1 == 0) & (col2 == 0)).sum()
    
    odds_ratio, p_value = fisher_exact([[a, b], [c, d]])
    
    results.append({
        'gene1': g1, 'gene2': g2, 'both': a, 'g1_only': b, 'g2_only': c, 'neither': d,
        'odds_ratio': odds_ratio, 'p_value': p_value
    })

results_df = pd.DataFrame(results)
print(f"Completed {len(results_df)} tests")

# Benjamini-Hochberg FDR correction
rejected, q_values, _, _ = multipletests(results_df['p_value'], method='fdr_bh', alpha=0.05)
results_df['q_value'] = q_values
results_df['significant'] = rejected

print(f"\nBH-corrected significant pairs: {results_df['significant'].sum()}")
print(f"\nTop 10 most significant:")
print(results_df.nsmallest(10, 'q_value')[['gene1', 'gene2', 'odds_ratio', 'q_value', 'significant']].round(4))



results_df['direction'] = results_df['odds_ratio'].apply(
    lambda x: 'Co-occurrence' if x > 1 else 'Mutual exclusivity'
)
sig = results_df[results_df['significant']].sort_values('q_value')
print(f"\nALL {sig.shape[0]} SIGNIFICANT PAIRS:")
print(sig[['gene1', 'gene2', 'odds_ratio', 'q_value', 'direction']].round(4))



TEST 2: ALL {len(top_genes)}C2 = {len(top_genes)*(len(top_genes)-1)//2} pairs
Completed 105 tests

BH-corrected significant pairs: 15

Top 10 most significant:
    gene1   gene2  odds_ratio  q_value  significant
2    TP53  CDKN2A     11.9641   0.0000         True
66  CSMD3    PCLO      3.0343   0.0018         True
92  SYNE1    PCLO      3.2667   0.0018         True
63  CSMD3   SYNE1      2.7351   0.0043         True
94  SYNE1   DNAH5      3.0694   0.0043         True
32   FAT1  NOTCH1      2.5348   0.0049         True
64  CSMD3   LRP1B      2.5918   0.0064         True
75  MUC16     FLG      2.8183   0.0064         True
18    TTN   MUC16      2.1360   0.0135         True
17    TTN   CSMD3      2.0661   0.0154         True

ALL 15 SIGNIFICANT PAIRS:
     gene1   gene2  odds_ratio  q_value      direction
2     TP53  CDKN2A     11.9641   0.0000  Co-occurrence
66   CSMD3    PCLO      3.0343   0.0018  Co-occurrence
92   SYNE1    PCLO      3.2667   0.0018  Co-occurrence
63   CSMD3   SYNE1  

In [6]:
tp53_pik3ca = results_df[(results_df['gene1'] == 'TP53') & (results_df['gene2'] == 'PIK3CA')]
print(tp53_pik3ca[['gene1', 'gene2', 'odds_ratio', 'p_value', 'q_value']].round(4))

  gene1   gene2  odds_ratio  p_value  q_value
6  TP53  PIK3CA      0.5289   0.0083   0.0536
